# Exploratory data analysis of SECOM

This notebook only explores; all logic lives in `src/secom`. Feature statistics are computed on the **training period only**, so no hold-out information can influence design decisions. The one exception is the label timeline in section 1, which shows class balance over the whole period and informs no preprocessing parameter.

In [1]:
from secom.data import load_secom
from secom.eda import (
    constant_feature_counts,
    correlated_pair_counts,
    missing_fraction,
    top_separating_features,
    weekly_fail_rate,
)
from secom.plots import (
    apply_style,
    plot_counts,
    plot_feature_distributions,
    plot_missing_histogram,
    plot_weekly_fail_rate,
    save_figure,
)
from secom.validation import temporal_split

apply_style()
features, labels, timestamps = load_secom()
split = temporal_split(timestamps)
train_features = features.iloc[split.train]
train_labels = labels.iloc[split.train]
print(f"Units: {features.shape[0]}, features: {features.shape[1]}")
print(f"Fails: {labels.sum()} ({labels.mean():.2%})")
print(f"Period: {timestamps.min()} to {timestamps.max()}")
print(f"Train: {len(split.train)} units, {train_labels.sum()} fails")
print(f"Hold-out: {len(split.test)} units, starting {timestamps.iloc[split.test[0]]}")

Units: 1567, features: 590
Fails: 104 (6.64%)
Period: 2008-07-19 11:55:00 to 2008-10-17 06:07:00
Train: 1254 units, 87 fails
Hold-out: 313 units, starting 2008-10-02 21:32:00


## 1. Class balance over time

In [2]:
weekly = weekly_fail_rate(labels, timestamps)
holdout_start = timestamps.iloc[split.test[0]]
save_figure(plot_weekly_fail_rate(weekly, holdout_start), "eda_fail_rate_over_time")
weekly

,units,fails,fail_rate
timestamp,,,
2008-07-20,13,3,0.230769
2008-07-27,21,2,0.095238
2008-08-03,48,10,0.208333
2008-08-10,108,14,0.129630
2008-08-17,51,7,0.137255
2008-08-24,208,22,0.105769
2008-08-31,169,7,0.041420
2008-09-07,133,2,0.015038
2008-09-14,95,4,0.042105


## 2. Missing values

In [3]:
missing = missing_fraction(train_features)
save_figure(plot_missing_histogram(missing, threshold=0.5), "eda_missing_values")
for level in (0.0, 0.1, 0.5, 0.9):
    print(f"Features with more than {level:.0%} missing: {(missing > level).sum()}")

Features with more than 0% missing: 442
Features with more than 10% missing: 52
Features with more than 50% missing: 24
Features with more than 90% missing: 4


## 3. Constant and near-constant features

In [4]:
constant_counts = constant_feature_counts(train_features)
labels_map = {
    "constant": "Constant",
    "near_constant": "One value in at least 95% of rows",
    "other": "Other",
}
figure = plot_counts(
    {labels_map[key]: value for key, value in constant_counts.items()},
    "Feature variability (training period)",
    "Features",
)
save_figure(figure, "eda_constant_features")
constant_counts

{'constant': 122, 'near_constant': 4, 'other': 464}

## 4. Correlation structure

In [5]:
pair_counts = correlated_pair_counts(train_features)
figure = plot_counts(
    {f"|r| at least {threshold}": count for threshold, count in pair_counts.items()},
    "Strongly correlated feature pairs (training period)",
    "Feature pairs",
)
save_figure(figure, "eda_correlated_pairs")
pair_counts

{0.8: 677, 0.9: 409, 0.95: 288, 0.99: 130}

## 5. Feature distributions by class

The four features with the largest standardized mean difference, among features with at least 30 observed values per class. With this many features, some separation arises by chance, so this view motivates feature selection inside the cross-validation and is not evidence of a real effect.

In [6]:
top_features = top_separating_features(train_features, train_labels)
figure = plot_feature_distributions(train_features[top_features], train_labels)
save_figure(figure, "eda_feature_distributions")
top_features

['f059', 'f431', 'f103', 'f510']

## Design decisions informed by this analysis

- Severe class imbalance rules out accuracy as the primary metric; PR-AUC is used.
- Many missing and constant features justify the drop, imputation and variance steps, which are all fitted inside the pipeline on training folds.
- Strong redundancy motivates the optional correlation filter and feature selection.
- Label rates vary over time, which is one reason to validate chronologically.